# 05 – Le cluster : Kafka à 3 brokers et Spark standalone

**Question :** que change le passage d'une seule machine à un cluster, et le système résiste-t-il à une panne ?

Le mode cluster est décrit dans `docker-compose.cluster.yml` et piloté par `scripts/cluster.sh` :
```bash
bash scripts/cluster.sh up       # 3 brokers Kafka + 1 master Spark + 2 workers + MongoDB
bash scripts/cluster.sh train    # entraînement sur le cluster
bash scripts/cluster.sh demo     # scoring temps réel sur le cluster
bash scripts/cluster.sh panne    # arrêt d'un broker pendant le flux
```
Ce notebook interroge le cluster **en marche** (lancer `bash scripts/cluster.sh up` avant de l'exécuter).

In [1]:
# Préparation : on travaille depuis la racine du projet pour importer src/ et config/
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E3E7EF", "axes.axisbelow": True})
AMBRE, ROUGE, TURQUOISE, GRIS = "#E89A00", "#E5384A", "#1BA597", "#C9D2E3"

In [2]:
import json, subprocess, urllib.request

def sh(cmd):
    """Exécute une commande (docker...) et renvoie sa sortie texte."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

print(sh("docker ps --format '{{.Names}}\t{{.Status}}' | grep momo | sort"))

momo-kafka-1	Up 25 minutes (healthy)
momo-kafka-2	Up 10 minutes (healthy)
momo-kafka-3	Up 25 minutes (healthy)
momo-mongo-express	Up 25 minutes
momo-mongo	Up 25 minutes
momo-spark-app	Up 14 minutes
momo-spark-master	Up 14 minutes
momo-spark-worker-1	Up 14 minutes
momo-spark-worker-2	Up 14 minutes



## 1. Kafka : 3 brokers, données répliquées
Le topic `momo.transactions.raw` est découpé en **3 partitions** (le travail est réparti) et chaque partition est copiée sur les **3 brokers** (`ReplicationFactor: 3`).
Chaque partition a un broker *leader* différent ; `Isr` liste les copies à jour. Avec `min.insync.replicas=2` et un producer en `acks=all`, une écriture n'est confirmée que lorsqu'au moins deux copies existent : la perte d'un broker ne perd aucun message.

In [3]:
print(sh("docker exec momo-kafka-1 /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka-1:19092 --describe --topic momo.transactions.raw"))

Topic: momo.transactions.raw	TopicId: ERcPXSuJQ6ifh6_jh9-AfA	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2
	Topic: momo.transactions.raw	Partition: 0	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 
	Topic: momo.transactions.raw	Partition: 1	Leader: 1	Replicas: 1,2,3	Isr: 1,3,2	Elr: 	LastKnownElr: 
	Topic: momo.transactions.raw	Partition: 2	Leader: 2	Replicas: 2,3,1	Isr: 3,1,2	Elr: 	LastKnownElr: 



## 2. Spark standalone : 1 master, 2 workers

In [4]:
etat = json.load(urllib.request.urlopen("http://localhost:8080/json/"))
print(f"Master : {etat['status']} · {etat['aliveworkers']} workers · {etat['cores']} cœurs · {etat['memory']} Mo")
pd.DataFrame([{"worker": f"worker-{i}", "état": w["state"], "cœurs": w["cores"], "mémoire (Mo)": w["memory"]}
              for i, w in enumerate(sorted(etat["workers"], key=lambda w: w["host"]), start=1)])

Master : ALIVE · 2 workers · 4 cœurs · 3072 Mo


,worker,état,cœurs,mémoire (Mo)
0,worker-1,ALIVE,2,1536
1,worker-2,ALIVE,2,1536


## 3. Entraînement : une machine contre le cluster
Même code (`src/ml/train_model.py`), même données, même forêt aléatoire. Seule change la valeur de `SPARK_MASTER` :
`local[*]` sur la machine, `spark://spark-master:7077` dans le conteneur `spark-app` (variable lue par `src/ml/spark.py`).

In [5]:
# Mesuré le 18/10 avec la commande complète « train_model --algo rf --test-data ... », même machine,
# conteneurs du cluster allumés dans les deux cas (une mesure de chaque : ordre de grandeur, pas un banc d'essai)
mesures = pd.DataFrame([
    {"mode": "local[*] : 8 cœurs, une seule JVM de 3 Go", "durée totale (s)": 403, "AUC-PR test": 0.801},
    {"mode": "cluster : 2 executors × 2 cœurs × 1 Go", "durée totale (s)": 224, "AUC-PR test": 0.800},
]).set_index("mode")
mesures

,durée totale (s),AUC-PR test
mode,,
"local[*] : 8 cœurs, une seule JVM de 3 Go",403,0.801
cluster : 2 executors × 2 cœurs × 1 Go,224,0.800


Même modèle (AUC-PR identique à la variation près). Le cluster termine **presque deux fois plus vite** alors qu'il utilise moins de cœurs (4 contre 8) :
en local, une seule JVM de 3 Go fait tout (lecture, cache, fenêtres, entraînement) et passe du temps à libérer sa mémoire ; sur le cluster,
le travail et les données sont répartis entre deux executors qui ont chacun leur mémoire.
Les deux workers partagent pourtant le même processeur : sur de vraies machines séparées, le gain serait plus net, et ajouter une machine ajoute des cœurs et de la mémoire sans changer une ligne de code. Le notebook 06 (PaySim, 6,36 millions de transactions) mesure ce que cela change sur un vrai volume.

## 4. Test de panne : un broker arrêté pendant le flux
Scénario (journal `logs/test_panne.log`) : le scoring tourne sur le cluster, le producer envoie 3 000 transactions à 20 par seconde ;
au bout de 40 s, le broker `kafka-2` est arrêté pendant 60 s, puis relancé.

In [6]:
print(open("logs/test_panne.log").read())

[02:24:36] consumer démarré
[02:24:36] producer démarré : 3000 transactions à 20 tx/s
[02:25:21] PANNE : kafka-2 arrêté
    	Topic: momo.transactions.raw	Partition: 0	Leader: 3	Replicas: 3,1,2	Isr: 3,1	Elr: 	LastKnownElr: 
    	Topic: momo.transactions.raw	Partition: 1	Leader: 1	Replicas: 1,2,3	Isr: 1,3	Elr: 	LastKnownElr: 
    	Topic: momo.transactions.raw	Partition: 2	Leader: 3	Replicas: 2,3,1	Isr: 3,1	Elr: 	LastKnownElr: 
[02:26:28] kafka-2 relancé
[02:27:09] producer terminé : TERMINÉ envoyés=3000 invalides=0 erreurs=0 durée=150.0s débit moyen=20.0/s
[02:27:35] consumer arrêté : Terminé : 3000 transactions écrites au total.
    	Topic: momo.transactions.raw	Partition: 0	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 
    	Topic: momo.transactions.raw	Partition: 1	Leader: 1	Replicas: 1,2,3	Isr: 1,3,2	Elr: 	LastKnownElr: 
    	Topic: momo.transactions.raw	Partition: 2	Leader: 3	Replicas: 2,3,1	Isr: 3,1,2	Elr: 	LastKnownElr: 
MongoDB brutes: 3000 | scorées: 3000 | doublons s

In [7]:
from pymongo import MongoClient
db = MongoClient("mongodb://localhost:27017")["momo_fraud"]
envoyees = 3000
brutes, scorees = db.transactions.count_documents({}), db.scored_transactions.count_documents({})
doublons = scorees - len(db.scored_transactions.distinct("transaction_id"))
pd.DataFrame([{"envoyées": envoyees, "stockées (brutes)": brutes, "scorées": scorees, "doublons": doublons}])

,envoyées,stockées (brutes),scorées,doublons
0,3000,3000,3000,0


## 5. Latence avant, pendant et après la panne
On classe chaque transaction selon l'heure à laquelle le producer l'a envoyée (`sent_at`), par rapport aux heures du journal.

In [8]:
import re
from datetime import datetime
journal = open("logs/test_panne.log").read()
jour = pd.Timestamp.now().date()
fuseau = datetime.now().astimezone().tzinfo   # le journal est à l'heure de la machine, sent_at en UTC
heure = lambda motif: pd.Timestamp(f"{jour} {re.search(r'\[(\d\d:\d\d:\d\d)\] ' + motif, journal).group(1)}").tz_localize(fuseau)
debut_panne, fin_panne = heure("PANNE"), heure("kafka-2 relancé")
docs = pd.DataFrame(list(db.scored_transactions.find({}, {"_id": 0, "sent_at": 1, "latency_ms": 1})))
docs["sent_at"] = pd.to_datetime(docs["sent_at"], utc=True)
docs["phase"] = pd.cut(docs["sent_at"], [pd.Timestamp.min.tz_localize("UTC"), debut_panne, fin_panne, pd.Timestamp.max.tz_localize("UTC")],
                       labels=["avant la panne", "pendant (kafka-2 arrêté)", "après la relance"])
(docs.groupby("phase", observed=True)["latency_ms"].describe(percentiles=[.5, .95])[["count", "50%", "95%", "max"]] / [1, 1000, 1000, 1000]).round(1)

,count,50%,95%,max
phase,,,,
avant la panne,837.0,34.4,52.4,54.5
pendant (kafka-2 arrêté),1340.0,8.4,17.2,20.6
après la relance,823.0,7.3,12.7,13.7


Lecture : *count* est en transactions, les autres colonnes en secondes.
- **Avant la panne**, la latence est élevée : c'est le **démarrage du scoring**. Les premiers micro-lots sont lents (chargement du modèle, compilation à la volée par la JVM, première jointure avec les profils) ; les transactions arrivent à 20 par seconde et s'accumulent, puis le retard se résorbe.
- **Pendant la panne**, la latence ne se dégrade pas : la bascule du leader de la partition 2 vers `kafka-3` est quasi immédiate, et le producer renvoie automatiquement les quelques messages refusés (`NotLeaderForPartitionError`, puis nouvel essai).
- **Une fois lancé**, le cluster score une transaction en 7 à 8 s en médiane (5,6 s en mode local : sur une seule machine, les échanges entre conteneurs ont un coût).

Conséquence pratique pour une démonstration : lancer le scoring une minute avant d'envoyer le flux.

## Ce qu'il faut retenir
- Le même code tourne en local ou sur le cluster : seule la variable `SPARK_MASTER` change.
- Kafka réplique chaque message sur 3 brokers : l'arrêt d'un broker en plein flux n'a fait perdre **aucune transaction** (3 000 envoyées, 3 000 scorées) ni dégradé la latence, et l'upsert sur `transaction_id` évite les doublons.
- Même sur une seule machine, répartir le travail entre deux executors a presque divisé par deux la durée d'entraînement (une mesure) ; l'intérêt principal reste la tolérance aux pannes et la possibilité d'ajouter des machines.